# CIFAR-10 FedAvg baseline: run on your computer

Open this notebook from inside the repository folder (the one that contains `configs/` and `src/`), then Run all.
It works on a CPU or a GPU, no Colab needed. It needs Python 3.9 or newer.

The steps run in order: check and install requirements, run the tests, download CIFAR-10 (about 170 MB, first run only) and train for 150 rounds, then show the results.
On a CPU expect roughly 15 to 60 minutes (an estimate).

In [ ]:
# 1. Locate the repository and install requirements if torch is missing
import subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir() and (ROOT / "configs").is_dir(), f"Open this notebook inside the repo folder, not {ROOT}"
print("Repository:", ROOT)

if sys.version_info < (3, 9):
    raise SystemExit(f"Python 3.9+ is needed; this is {sys.version.split()[0]}.")
try:
    import torch, torchvision
    print("torch", torch.__version__, "already installed")
except ImportError:
    print("Installing requirements (torch can take a few minutes)...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT / "requirements.txt")],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:])
        raise SystemExit("pip install failed; see the output above.")
    print("requirements installed")

In [ ]:
# 2. Unit tests. If they fail, the baseline is not run.
r = subprocess.run([sys.executable, "-m", "pytest", "-q", str(ROOT / "tests")],
                   capture_output=True, text=True, cwd=ROOT)
print(r.stdout[-3000:])
if r.returncode != 0:
    print(r.stderr[-2000:])
    raise SystemExit("Tests failed. Fix them or send me this output.")

In [ ]:
# 3. CIFAR-10 baseline: 50 clients, 10 per round, Dirichlet alpha=0.1, 150 rounds
import os
sys.path.insert(0, str(ROOT / "experiments" / "baseline"))
os.chdir(ROOT)
import run_baseline as rb

ALPHA = "0.1"  # 0.1 strongly non-IID, 0.5 moderate, 10 near-IID
saved_argv = sys.argv
sys.argv = ["notebook", "--config", str(ROOT / "configs" / "experiment_cifar10.yaml"), "--alpha", ALPHA]
try:
    args = rb.parse_args()
finally:
    sys.argv = saved_argv
run_dir = rb.run(rb.build_config(args))

In [ ]:
# 4. Results: accuracy, simulated federation totals, and this machine's energy
import json
summary = json.loads((run_dir / "summary.json").read_text())
energy = json.loads((run_dir / "energy.json").read_text())
print(f"Final accuracy : {summary['final_accuracy']:.4f} (best {summary['best_accuracy']:.4f})")
print(f"Simulated federation : {summary['sim_total_latency_s'] / 60:.1f} min, "
      f"{summary['sim_total_energy_kwh'] * 1000:.2f} Wh, {summary['sim_total_carbon_gco2e']:.2f} gCO2e")
print(f"This machine's energy : {energy['energy_kwh'] * 1000:.2f} Wh, "
      f"CPU [{energy['cpu_source']}], GPU [{energy['gpu_source']}]")
print("Saved in:", run_dir)